In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os


# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!wget https://raw.githubusercontent.com/Sanjidx090/ytranscript/refs/heads/main/tnp/videos_with_bangla.csv

--2026-03-26 18:29:44--  https://raw.githubusercontent.com/Sanjidx090/ytranscript/refs/heads/main/tnp/videos_with_bangla.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 527 [text/plain]
Saving to: ‘videos_with_bangla.csv’

videos_with_bangla. 100%[===================>]     527  --.-KB/s    in 0s      

2026-03-26 18:29:44 (14.0 MB/s) - ‘videos_with_bangla.csv’ saved [527/527]



In [3]:
!pip install yt-dlp pandas tqdm huggingface_hub datasets
!sudo apt-get install ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 71.3 MB/s eta 0:00:00



ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 138 not upgraded.


In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_Token")


In [5]:
#####This one has somee premium 
"""
Lipighor — Batch Stream-Upload Pipeline
========================================
Flow:
  1. Pull manifest.csv from HuggingFace (single source of truth)
  2. Work through videos in batches of BATCH_SIZE
  3. Per video : download → upload audio → delete local file
  4. After each batch : update manifest on HF
  5. Repeat until all videos are done

Skip logic (never retried once set):
  - status == "success"     → already on HF
  - status == "unavailable" → video deleted/private on YouTube

Re-tried automatically:
  - status == "failed"      → transient error, will retry
  - not in manifest at all  → fresh

Usage on Kaggle:
    Add secret  HF_Token = hf_xxxx
    then:  %run lipighor_upload.py
"""

import argparse, csv, io, os, sys, time, threading, subprocess
from pathlib import Path

# ── auto-install ──────────────────────────────────────────────────────────────
for _pkg in ("yt_dlp", "huggingface_hub", "pandas"):
    try:
        __import__(_pkg)
    except ImportError:
        print(f"[setup] installing {_pkg} …")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

import pandas as pd
from huggingface_hub import HfApi, create_repo

# ── configuration ─────────────────────────────────────────────────────────────
DEFAULT_CSV    = "videos_with_bangla.csv"
HF_REPO        = "frovolts/ssdata"
HF_REPO_TYPE   = "dataset"
HF_MANIFEST    = "data/manifest.csv"          # path inside the HF repo
HF_AUDIO_DIR   = "data/audio"                 # path inside the HF repo

TEMP_DIR       = Path("/tmp/lipighor_audio")

SAMPLE_RATE    = 16_000
CHANNELS       = 1
BATCH_SIZE     = 10                           # upload manifest every N videos

MAX_DL_RETRIES = 3
DL_RETRY_DELAY = 5

HF_INIT_WAIT   = 10
HF_MAX_WAIT    = 300
HF_MAX_RETRIES = 6

MANIFEST_COLS  = ["video_id", "status", "hf_path", "error", "timestamp"]

# statuses that are FINAL — never retried
FINAL_STATUSES = {"success", "unavailable"}
# ── end configuration ─────────────────────────────────────────────────────────


def ts() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")


# ── token ─────────────────────────────────────────────────────────────────────

def get_token() -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_Token").strip()
        if token:
            return token
    except Exception:
        pass
    token = os.environ.get("HF_Token", "").strip()
    if token:
        return token
    sys.exit(
        "[ERROR] Token not found.\n"
        "  Kaggle : Add-ons → Secrets → HF_Token\n"
        "  Local  : export HF_Token=hf_xxxx"
    )


# ── manifest (lives on HF) ────────────────────────────────────────────────────

def pull_manifest(api: HfApi, token: str) -> pd.DataFrame:
    """Download manifest.csv from HF. Returns empty DataFrame if not found."""
    try:
        content = api.hf_hub_download(
            repo_id=HF_REPO, repo_type=HF_REPO_TYPE,
            filename=HF_MANIFEST, token=token,
        )
        df = pd.read_csv(content)
        print(f"[Manifest] Pulled from HF — {len(df)} rows")
        return df
    except Exception:
        print("[Manifest] Not found on HF — starting fresh")
        return pd.DataFrame(columns=MANIFEST_COLS)


def push_manifest(api: HfApi, token: str, df: pd.DataFrame) -> None:
    """Upload the in-memory manifest back to HF."""
    buf = io.BytesIO(df.to_csv(index=False).encode())
    _hf_upload_with_backoff(
        api, token,
        fileobj=buf,
        path_in_repo=HF_MANIFEST,
        commit_message=f"Update manifest ({len(df)} rows)",
    )
    print(f"[Manifest] Pushed to HF — {len(df)} rows")


def done_ids(df: pd.DataFrame) -> set:
    """Video IDs that are in a FINAL state and should never be retried."""
    return set(df[df["status"].isin(FINAL_STATUSES)]["video_id"])


# ── HF upload with back-off ───────────────────────────────────────────────────

def _hf_upload_with_backoff(api, token, *, fileobj=None, local_path=None,
                             path_in_repo, commit_message):
    wait = HF_INIT_WAIT
    kwargs = dict(
        path_in_repo=path_in_repo,
        repo_id=HF_REPO,
        repo_type=HF_REPO_TYPE,
        token=token,
        commit_message=commit_message,
    )
    for attempt in range(1, HF_MAX_RETRIES + 1):
        try:
            if fileobj is not None:
                fileobj.seek(0)
                api.upload_file(path_or_fileobj=fileobj, **kwargs)
            else:
                api.upload_file(path_or_fileobj=str(local_path), **kwargs)
            return
        except Exception as e:
            is_rate = any(k in str(e).lower() for k in ("429", "rate limit", "too many"))
            if attempt == HF_MAX_RETRIES:
                raise RuntimeError(f"HF upload failed after {HF_MAX_RETRIES} attempts: {e}")
            wait_time = wait if is_rate else 5
            print(f"  [HF {'rate-limit' if is_rate else 'error'}] waiting {wait_time}s "
                  f"(attempt {attempt}/{HF_MAX_RETRIES}) …")
            time.sleep(wait_time)
            if is_rate:
                wait = min(wait * 2, HF_MAX_WAIT)


def upload_audio(api, token, local_path: Path) -> str:
    """Upload one WAV and return its HF path."""
    hf_path = f"{HF_AUDIO_DIR}/{local_path.name}"
    _hf_upload_with_backoff(
        api, token,
        local_path=local_path,
        path_in_repo=hf_path,
        commit_message=f"Add {local_path.stem}",
    )
    return hf_path


# ── download ──────────────────────────────────────────────────────────────────

UNAVAILABLE_PHRASES = (
    "video is not available",
    "video unavailable",
    "private video",
    "has been removed",
    "account associated",
)

def download_audio(video_id: str) -> Path:
    """
    Download best audio → 16kHz mono WAV in TEMP_DIR.
    Raises RuntimeError (unavailable) or generic error.
    """
    out_path = TEMP_DIR / f"{video_id}.wav"
    tmp_tmpl = TEMP_DIR / f"{video_id}.%(ext)s"
    url      = f"https://www.youtube.com/watch?v={video_id}"

    cmd = [
        sys.executable, "-m", "yt_dlp",
        "--format", "bestaudio/best",
        "--extract-audio", "--audio-format", "wav",
        "--output", str(tmp_tmpl),
        "--postprocessor-args", f"ffmpeg:-ar {SAMPLE_RATE} -ac {CHANNELS}",
        "--no-playlist", "--quiet", "--no-warnings",
        url,
    ]

    err = "unknown error"
    for attempt in range(1, MAX_DL_RETRIES + 1):
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
            stderr = (r.stderr or r.stdout).strip()

            # Detect permanently unavailable videos — no point retrying
            if any(p in stderr.lower() for p in UNAVAILABLE_PHRASES):
                raise RuntimeError(f"UNAVAILABLE:{stderr}")

            if r.returncode == 0 and out_path.exists() and out_path.stat().st_size > 0:
                return out_path

            err = stderr or "yt-dlp returned non-zero"
        except RuntimeError:
            raise   # propagate UNAVAILABLE immediately
        except subprocess.TimeoutExpired:
            err = "timed out after 300s"

        if attempt < MAX_DL_RETRIES:
            time.sleep(DL_RETRY_DELAY * attempt)

    raise RuntimeError(err)


# ── per-video pipeline ────────────────────────────────────────────────────────

def process_video(video_id: str, api: HfApi, token: str) -> dict:
    local_wav = None
    try:
        local_wav = download_audio(video_id)
        hf_path   = upload_audio(api, token, local_wav)
        return {"video_id": video_id, "status": "success",
                "hf_path": hf_path, "error": "", "timestamp": ts()}

    except RuntimeError as e:
        if str(e).startswith("UNAVAILABLE:"):
            status = "unavailable"
            err    = str(e)[len("UNAVAILABLE:"):]
        else:
            status = "failed"
            err    = str(e)
        return {"video_id": video_id, "status": status,
                "hf_path": "", "error": err, "timestamp": ts()}

    finally:
        if local_wav and local_wav.exists():
            local_wav.unlink()


# ── main ──────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--csv",        default=DEFAULT_CSV)
    parser.add_argument("--batch-size", type=int, default=BATCH_SIZE)
    parser.add_argument("--public",     action="store_true", help="Make repo public (default: private)")
    args, _ = parser.parse_known_args()   # ignore Jupyter kernel args

    TEMP_DIR.mkdir(parents=True, exist_ok=True)
    token = get_token()
    api   = HfApi()

    # ── repo setup ────────────────────────────────────────────────────────────
    private = not args.public
    create_repo(repo_id=HF_REPO, repo_type=HF_REPO_TYPE,
                private=private, exist_ok=True, token=token)
    print(f"[HF]  Repo → https://huggingface.co/datasets/{HF_REPO}")

    # ── pull manifest from HF ─────────────────────────────────────────────────
    manifest_df = pull_manifest(api, token)
    skip        = done_ids(manifest_df)

    # ── load video list ───────────────────────────────────────────────────────
    df = pd.read_csv(args.csv)
    if "has_bangla" in df.columns:
        df = df[df["has_bangla"] == True]
    all_ids   = df["video_id"].dropna().unique().tolist()
    remaining = [v for v in all_ids if v not in skip]

    print(f"[CSV] Total        : {len(all_ids)}")
    print(f"[CSV] Done (final) : {len(skip)}")
    print(f"[CSV] To process   : {len(remaining)}\n")

    if not remaining:
        print("[DONE] All videos processed.")
        return

    # ── batch loop ────────────────────────────────────────────────────────────
    new_rows = []
    total    = len(all_ids)
    done_n   = len(skip)

    def flush_batch(label="batch"):
        """Merge new_rows into manifest and push to HF."""
        nonlocal manifest_df, new_rows
        if not new_rows:
            return
        manifest_df = pd.concat(
            [manifest_df, pd.DataFrame(new_rows)], ignore_index=True
        )
        push_manifest(api, token, manifest_df)
        print(f"  [saved {len(new_rows)} rows — {label}]")
        new_rows = []

    def print_summary():
        counts = manifest_df["status"].value_counts()
        print("\n" + "─" * 45)
        for status, count in counts.items():
            icon = "✓" if status == "success" \
                   else ("✗✗" if status == "unavailable" else "✗")
            print(f"  {icon}  {status:<20} : {count}")
        print("─" * 45)

        failed = manifest_df[manifest_df["status"] == "failed"]
        if not failed.empty:
            print("\n[WILL RETRY NEXT RUN]")
            for _, r in failed.iterrows():
                print(f"  {r['video_id']}  →  {r['error'][:80]}")

        unavail = manifest_df[manifest_df["status"] == "unavailable"]
        if not unavail.empty:
            print("\n[PERMANENTLY UNAVAILABLE — skipped forever]")
            for _, r in unavail.iterrows():
                print(f"  {r['video_id']}")

        print(f"\n[HF] https://huggingface.co/datasets/{HF_REPO}")

    try:
        for i, video_id in enumerate(remaining):
            res     = process_video(video_id, api, token)
            done_n += 1
            new_rows.append(res)

            icon = "✓" if res["status"] == "success" \
                   else ("✗✗" if res["status"] == "unavailable" else "✗")
            print(f"[{done_n:>3}/{total}] {icon}  {video_id}  ({res['status']})")

            # push manifest every BATCH_SIZE videos or on the last one
            if len(new_rows) % args.batch_size == 0 or i == len(remaining) - 1:
                flush_batch(label=f"every {args.batch_size}")

    except KeyboardInterrupt:
        print("\n\n[INTERRUPTED] Saving progress before exit …")
        flush_batch(label="interrupt save")
        print_summary()
        print("[SAFE TO RESTART] Run the script again to continue.\n")
        return   # clean exit, no traceback

    print_summary()
    print("[DONE]")


if __name__ == "__main__":
    main()

[HF]  Repo → https://huggingface.co/datasets/frovolts/ssdata
[Manifest] Not found on HF — starting fresh
[CSV] Total        : 9
[CSV] Done (final) : 0
[CSV] To process   : 9



Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[  1/9] ✓  wLR0crYMXUk  (success)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

[  8/9] ✓  X020uvCGuN8  (success)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[  9/9] ✓  7eEUCYhFxeo  (success)
[Manifest] Pushed to HF — 9 rows
  [saved 9 rows — every 10]

─────────────────────────────────────────────
  ✓  success              : 8
  ✗✗  unavailable          : 1
─────────────────────────────────────────────

[PERMANENTLY UNAVAILABLE — skipped forever]
  MFH3wO6n-Sk

[HF] https://huggingface.co/datasets/frovolts/ssdata
[DONE]


In [6]:
import json
import os
import glob
import shutil
import re
import html
import time
import tempfile
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Tuple

import yt_dlp
from huggingface_hub import HfApi, hf_hub_download, list_repo_files, upload_folder

# ══════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════
# Replace 'your_token_here' with your actual HF write token
HF_TOKEN          = secret_value_0
REPO_ID           = "frovolts/ssdata"
SOURCE_DIR        = "data/audio"         # Folder containing the source files
TARGET_DIR        = "asr_train_ready"    # New folder to create for output
LOCAL_WORK_DIR    = Path("asr_workspace") # Local scratch folder

# Chunking settings
MIN_SEC            = 18        # minimum chunk duration (seconds)
MAX_SEC            = 30        # maximum chunk duration (seconds)
SILENCE_THRESHOLD  = 0.500     # gap (seconds) signaling sentence boundary

# Safety: stop fetching if blocked this many times in a row
MAX_CONSECUTIVE_BLOCKS = 15
REPROCESS_EXISTING = False
LANG_CODE = "bn"
# ══════════════════════════════════════════════════════


def clean_text(text: str) -> str:
    text = html.unescape(text)
    text = text.replace('>>', ' ').replace('>', ' ')
    text = re.sub(r'[।?!,;:\-—"\'\(\)\[\]\{\}]', ' ', text)
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


class TranscriptFetcher:
    LANGS = ["bn", "bn-IN", "bn-BD", "bn-Beng"]

    def fetch(self, video_id: str) -> Tuple[Optional[List[Dict]], str]:
        url = f"https://www.youtube.com/watch?v={video_id}"
        with tempfile.TemporaryDirectory() as tmpdir:
            for auto in [False, True]:
                result = self._download(url, tmpdir, video_id, auto)
                if result == "blocked":
                    return None, "blocked"
                if result:
                    return result, "auto-generated" if auto else "manual"
        return None, "none"

    def _download(self, url, tmpdir, video_id, auto):
        opts = {
            "skip_download":    True,
            "writesubtitles":   not auto,
            "writeautomaticsub": auto,
            "subtitleslangs":   self.LANGS,
            "subtitlesformat":  "json3",
            "outtmpl":          os.path.join(tmpdir, "%(id)s.%(ext)s"),
            "quiet":            True,
            "no_warnings":      True,
            "retries":          3,
            "socket_timeout":   30,
            "http_headers": {
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
            },
        }
        try:
            with yt_dlp.YoutubeDL(opts) as ydl:
                ydl.extract_info(url, download=True)
            files = glob.glob(os.path.join(tmpdir, f"{video_id}.*.json3"))
            if not files:
                return None
            with open(files[0], "r", encoding="utf-8") as f:
                return self._parse_words(json.load(f))
        except Exception as e:
            msg = str(e).lower()
            if any(k in msg for k in ["429", "too many requests", "blocked", "bot"]):
                return "blocked"
            return None

    def _parse_words(self, raw: dict) -> Optional[List[Dict]]:
        words = []
        for event in raw.get("events", []):
            if "segs" not in event: continue
            base_start_ms = int(event.get("tStartMs", 0))
            current_offset = 0
            for seg in event["segs"]:
                word_text = seg.get("utf8", "").strip()
                if not word_text or word_text == "\n": continue
                t_offset = int(seg.get("tOffsetMs", current_offset))
                word_start_ms = base_start_ms + t_offset
                word_end_ms   = word_start_ms + 500
                current_offset = t_offset + 500
                words.append({
                    "text":  word_text,
                    "start": word_start_ms / 1000.0,
                    "end":   word_end_ms   / 1000.0,
                })
        return words if words else None


def chunk_words(words: List[Dict]) -> List[Dict]:
    if not words: return []
    chunks, buffer, chunk_id = [], [], 1

    def flush(buf):
        nonlocal chunk_id
        if not buf: return
        raw_text = " ".join(w["text"] for w in buf)
        clean_txt = clean_text(raw_text)
        if len(clean_txt) < 2: return
        chunks.append({
            "id": chunk_id,
            "start": round(buf[0]["start"], 3),
            "end": round(buf[-1]["end"], 3),
            "text": clean_txt,
        })
        chunk_id += 1

    i = 0
    while i < len(words):
        word = words[i]
        buffer.append(word)
        dur = buffer[-1]["end"] - buffer[0]["start"]

        if dur < MIN_SEC:
            i += 1
            continue

        if MIN_SEC <= dur <= MAX_SEC:
            is_sentence_end = False
            if i + 1 < len(words):
                gap = words[i + 1]["start"] - buffer[-1]["end"]
                if gap > SILENCE_THRESHOLD: is_sentence_end = True
            else:
                is_sentence_end = True

            if is_sentence_end or dur > (MAX_SEC - 1):
                flush(buffer)
                buffer = []
        elif dur > MAX_SEC:
            last_word = buffer.pop()
            flush(buffer)
            buffer = [last_word]
        i += 1

    if buffer: flush(buffer)
    return chunks


def get_video_ids_from_hf(api: HfApi) -> List[str]:
    """Lists video IDs by scanning any files in SOURCE_DIR on HF."""
    print(f"🔍 Listing files in {REPO_ID}/{SOURCE_DIR} ...")
    try:
        all_files = list(list_repo_files(REPO_ID, repo_type="dataset", token=HF_TOKEN))
        ids = []
        # Normalizing path for comparison
        prefix = SOURCE_DIR.strip("/") + "/"
        
        for f in all_files:
            if f.startswith(prefix):
                # Get filename (e.g., 'abc12345.mp3' or 'abc12345_unified.json')
                basename = os.path.basename(f)
                if not basename: continue
                
                # Split at the FIRST underscore or dot to isolate the ID
                # Example: 'videoID_unified.json' -> 'videoID'
                # Example: 'videoID.mp3' -> 'videoID'
                video_id = re.split(r'[_.]', basename)[0]
                
                if video_id and video_id not in ids:
                    ids.append(video_id)
        
        print(f"   ✅ Found {len(ids)} unique video IDs.")
        return ids
    except Exception as e:
        print(f"❌ Error listing HF files: {e}")
        return []


def get_already_done(api: HfApi) -> set:
    try:
        all_files = list(list_repo_files(REPO_ID, repo_type="dataset", token=HF_TOKEN))
        prefix = TARGET_DIR.strip("/") + "/"
        return {os.path.basename(f).replace("_asr.json", "") for f in all_files if f.startswith(prefix) and f.endswith("_asr.json")}
    except:
        return set()


def main():
    api = HfApi(token=HF_TOKEN)
    fetcher = TranscriptFetcher()

    if LOCAL_WORK_DIR.exists(): shutil.rmtree(LOCAL_WORK_DIR)
    LOCAL_WORK_DIR.mkdir(parents=True)

    video_ids = get_video_ids_from_hf(api)
    if not video_ids:
        print("❌ No video IDs found. Check if the folder 'data/audio' has files and your HF_TOKEN is correct.")
        return

    if not REPROCESS_EXISTING:
        already_done = get_already_done(api)
        video_ids = [v for v in video_ids if v not in already_done]
        print(f"⏭️  Skipping {len(already_done)} already-processed. Remaining: {len(video_ids)}\n")

    stats = {"total": 0, "success": 0, "no_transcript": 0, "blocked": 0, "empty_chunks": 0}
    block_count = 0

    for idx, video_id in enumerate(video_ids, 1):
        if block_count >= MAX_CONSECUTIVE_BLOCKS:
            print(f"\n🚫 Block limit reached ({MAX_CONSECUTIVE_BLOCKS}) — stopping.")
            break

        stats["total"] += 1
        print(f"[{idx}/{len(video_ids)}] {video_id} ...", end=" ", flush=True)

        words, src = fetcher.fetch(video_id)
        if src == "blocked":
            block_count += 1
            stats["blocked"] += 1
            print(f"🚫 BLOCKED ({block_count})")
            time.sleep(5 * block_count)
            continue

        block_count = 0 
        if not words:
            stats["no_transcript"] += 1
            print("❌ No Bengali transcript.")
            continue

        chunks = chunk_words(words)
        if not chunks:
            stats["empty_chunks"] += 1
            print("⚠️ No valid chunks.")
            continue

        total_dur = sum(c["end"] - c["start"] for c in chunks)
        output = {
            "video_id": video_id,
            "metadata": {
                "source": src,
                "total_chunks": len(chunks),
                "total_duration_sec": round(total_dur, 2),
                "created_at": datetime.now().isoformat(),
            },
            "segments": chunks,
        }

        with open(LOCAL_WORK_DIR / f"{video_id}_asr.json", "w", encoding="utf-8") as f:
            json.dump(output, f, ensure_ascii=False, indent=2)

        stats["success"] += 1
        print(f"✅ {src} — {len(chunks)} chunks | {total_dur:.1f}s")

    saved = list(LOCAL_WORK_DIR.glob("*.json"))
    if saved:
        print(f"\n📦 Uploading {len(saved)} files to {TARGET_DIR}/ ...")
        try:
            upload_folder(
                folder_path=str(LOCAL_WORK_DIR),
                path_in_repo=TARGET_DIR,
                repo_id=REPO_ID,
                repo_type="dataset",
                token=HF_TOKEN,
                commit_message=f"Add ASR chunks for {stats['success']} videos"
            )
            print("💾 Upload successful!")
        except Exception as e:
            print(f"⚠️ Upload failed: {e}")

    print(f"\n🎉 Done! Success: {stats['success']} | No Transcript: {stats['no_transcript']} | Blocked: {stats['blocked']}")


if __name__ == "__main__":
    main()

🔍 Listing files in frovolts/ssdata/data/audio ...
   ✅ Found 8 unique video IDs.
⏭️  Skipping 0 already-processed. Remaining: 8

✅ auto-generated — 57 chunks | 1447.4s
[2/8] 8c7IdHtZt ... 

ERROR: [youtube:truncated_id] 8c7IdHtZt: Incomplete YouTube ID 8c7IdHtZt. URL https://www.youtube.com/watch?v=8c7IdHtZt looks truncated.
ERROR: [youtube:truncated_id] 8c7IdHtZt: Incomplete YouTube ID 8c7IdHtZt. URL https://www.youtube.com/watch?v=8c7IdHtZt looks truncated.


❌ No Bengali transcript.
✅ auto-generated — 50 chunks | 1129.9s
✅ auto-generated — 97 chunks | 2036.4s
✅ manual — 2 chunks | 19.6s
✅ auto-generated — 50 chunks | 1151.4s
✅ auto-generated — 50 chunks | 1150.0s
✅ auto-generated — 20 chunks | 533.1s

📦 Uploading 7 files to asr_train_ready/ ...
💾 Upload successful!

🎉 Done! Success: 7 | No Transcript: 1 | Blocked: 0


In [7]:
import argparse
import json
import os
import csv
import shutil
import time
from pathlib import Path

from huggingface_hub import hf_hub_download, list_repo_files, HfApi
from pydub import AudioSegment
from tqdm import tqdm

# ══════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════
REPO_ID          = "frovolts/ssdata"
ASR_JSON_DIR     = "asr_train_ready"
AUDIO_DIR        = "data/audio"
OUTPUT_DIR       = "wavs_asr_chunks"
SAMPLE_RATE      = 16000
CHANNELS         = 1
HF_TOKEN         = secret_value_0
HF_UPLOAD_REPO   = "frovolts/ssdata"
HF_UPLOAD_PATH   = "wavs_asr_chunks"
UPLOAD_RETRIES   = 3
RETRY_DELAY      = 5
# ══════════════════════════════════════════════════════

api = HfApi()

def check_ffmpeg():
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("❌ ffmpeg not found! Install via 'apt-get install ffmpeg'.")

def get_manifest_paths(video_filter=None):
    print(f"🔍 Scanning {REPO_ID}/{ASR_JSON_DIR} for manifests...")
    all_files = list(list_repo_files(REPO_ID, repo_type="dataset", token=HF_TOKEN))
    paths = [f for f in all_files if f.startswith(ASR_JSON_DIR + "/") and f.endswith("_asr.json")]
    if video_filter:
        paths = [p for p in paths if any(v in p for v in video_filter)]
    print(f"   ✅ Found {len(paths)} manifests.")
    return paths, all_files

def download_json(remote_path):
    local = hf_hub_download(repo_id=REPO_ID, filename=remote_path, repo_type="dataset", token=HF_TOKEN)
    with open(local, "r", encoding="utf-8") as f:
        return json.load(f)

def find_audio_path(video_id, all_files):
    for ext in ["mp3", "wav", "m4a", "opus"]:
        candidate = f"{AUDIO_DIR}/{video_id}.{ext}"
        if candidate in all_files:
            return candidate
    return None

def load_audio(remote_path):
    local = hf_hub_download(repo_id=REPO_ID, filename=remote_path, repo_type="dataset", token=HF_TOKEN)
    return AudioSegment.from_file(local).set_frame_rate(SAMPLE_RATE).set_channels(CHANNELS)

def chunk_and_save(audio, segments, video_id, wav_dir, total_sec, max_sec):
    metadata_rows = []
    added = 0.0

    for seg in segments:
        if max_sec and (total_sec + added) >= max_sec:
            break
        start_ms = int(seg["start"] * 1000)
        end_ms   = int(seg["end"]   * 1000)
        text = seg["text"].strip()
        if not text or end_ms <= start_ms:
            continue

        chunk_name = f"{video_id}_chunk_{seg['id']:04d}.wav"
        chunk_path = wav_dir / chunk_name
        audio[start_ms : end_ms + 50].export(str(chunk_path), format="wav")

        duration = (end_ms - start_ms) / 1000.0
        added += duration
        metadata_rows.append({
            "file_name": f"wavs/{chunk_name}",
            "duration":  round(duration, 3),
            "text":      text,
            "video_id":  video_id,
            "chunk_id":  seg["id"],
        })
    return metadata_rows, added

from huggingface_hub import hf_hub_download, list_repo_files, HfApi, CommitOperationAdd  # add CommitOperationAdd

# ── replace upload_video_chunks ──────────────────────────────────────────────
def upload_video_chunks(video_id, wav_dir, meta_rows, failed_log_path):
    chunk_files = [wav_dir / Path(r["file_name"]).name for r in meta_rows]

    shard_path = wav_dir.parent / f"_shard_{video_id}.csv"
    headers = ["file_name", "duration", "text", "video_id", "chunk_id"]
    with open(shard_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        writer.writerows(meta_rows)

    files_to_upload = chunk_files + [shard_path]

    def _dest(p: Path):
        return (
            f"{HF_UPLOAD_PATH}/wavs/{p.name}"
            if p.suffix == ".wav"
            else f"{HF_UPLOAD_PATH}/shards/{p.name}"
        )

    operations = [
        CommitOperationAdd(path_in_repo=_dest(p), path_or_fileobj=str(p))
        for p in files_to_upload
    ]

    for attempt in range(1, UPLOAD_RETRIES + 1):
        try:
            api.create_commit(
                repo_id=HF_UPLOAD_REPO,
                repo_type="dataset",
                token=HF_TOKEN,
                commit_message=f"chunks: {video_id} ({len(chunk_files)} wavs)",
                operations=operations,
            )
            # single verification call
            remote = set(list_repo_files(HF_UPLOAD_REPO, repo_type="dataset", token=HF_TOKEN))
            missing = [op.path_in_repo for op in operations if op.path_in_repo not in remote]
            if not missing:
                for p in files_to_upload:
                    p.unlink(missing_ok=True)
                return True
            print(f"   ⚠️  {len(missing)} files missing after commit (attempt {attempt})")
        except Exception as e:
            print(f"   ❌ Batch commit error attempt {attempt}/{UPLOAD_RETRIES}: {e}")
        if attempt < UPLOAD_RETRIES:
            time.sleep(RETRY_DELAY)

    # log everything that failed
    with open(failed_log_path, "a", encoding="utf-8") as f:
        for p in files_to_upload:
            f.write(f"{p}|{_dest(p)}\n")
    print(f"   🔴 Permanently failed batch for {video_id}")
    return False
    
def append_metadata(meta_rows, out_dir):
    csv_path = out_dir / "metadata.csv"
    headers = ["file_name", "duration", "text", "video_id", "chunk_id"]
    mode = "a" if csv_path.exists() else "w"
    with open(csv_path, mode, newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        if mode == "w":
            writer.writeheader()
        writer.writerows(meta_rows)

def upload_master_metadata(out_dir):
    csv_path = out_dir / "metadata.csv"
    if not csv_path.exists():
        return
    try:
        api.upload_file(
            path_or_fileobj=str(csv_path),
            path_in_repo=f"{HF_UPLOAD_PATH}/metadata.csv",
            repo_id=HF_UPLOAD_REPO,
            repo_type="dataset",
            token=HF_TOKEN,
        )
    except Exception as e:
        print(f"   ⚠️  Could not update master metadata.csv: {e}")

# ── replace retry_failed_uploads ─────────────────────────────────────────────
def retry_failed_uploads(failed_log_path):
    if not failed_log_path.exists():
        return
    with open(failed_log_path, "r", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]
    if not lines:
        return

    print(f"\n🔁 Retrying {len(lines)} previously failed files in one batch...")

    available, missing_local = [], []
    for line in lines:
        local_str, dest = line.split("|", 1)
        p = Path(local_str)
        if p.exists():
            available.append((p, dest))
        else:
            print(f"   ⚠️  Local file gone, skipping: {local_str}")
            missing_local.append(line)

    if not available:
        with open(failed_log_path, "w", encoding="utf-8") as f:
            f.write("\n".join(missing_local) + ("\n" if missing_local else ""))
        return

    operations = [
        CommitOperationAdd(path_in_repo=dest, path_or_fileobj=str(p))
        for p, dest in available
    ]

    try:
        api.create_commit(
            repo_id=HF_UPLOAD_REPO,
            repo_type="dataset",
            token=HF_TOKEN,
            commit_message=f"retry: {len(operations)} failed uploads",
            operations=operations,
        )
        remote = set(list_repo_files(HF_UPLOAD_REPO, repo_type="dataset", token=HF_TOKEN))
        still_missing = [(p, dest) for p, dest in available if dest not in remote]

        for p, dest in available:
            if (p, dest) not in still_missing:
                p.unlink(missing_ok=True)

        remaining_lines = missing_local + [f"{p}|{dest}" for p, dest in still_missing]
        with open(failed_log_path, "w", encoding="utf-8") as f:
            f.write("\n".join(remaining_lines) + ("\n" if remaining_lines else ""))

        if still_missing:
            print(f"   ⚠️  {len(still_missing)} files still failed. See: {failed_log_path}")
        else:
            print("   🎉 All retries succeeded!")
    except Exception as e:
        print(f"   ❌ Retry batch commit failed: {e}")
        
def get_done_ids_from_hf():
    """Check HF remote metadata.csv (not local) to determine already-processed videos."""
    done_ids = set()
    try:
        remote_files = list(list_repo_files(HF_UPLOAD_REPO, repo_type="dataset", token=HF_TOKEN))
        remote_meta  = f"{HF_UPLOAD_PATH}/metadata.csv"
        if remote_meta in remote_files:
            local_meta = hf_hub_download(
                repo_id=HF_UPLOAD_REPO,
                filename=remote_meta,
                repo_type="dataset",
                token=HF_TOKEN,
                force_download=True,
            )
            with open(local_meta, "r", encoding="utf-8") as f:
                done_ids = {row["video_id"] for row in csv.DictReader(f)}
            print(f"   ♻️  Resuming — {len(done_ids)} videos already on HF, skipping them.")
        else:
            print("   🆕 No remote metadata found, starting fresh.")
    except Exception as e:
        print(f"   ⚠️  Could not fetch remote metadata for resume check: {e}")
    return done_ids

def main(video_filter=None, max_hours=None):
    check_ffmpeg()
    out_dir = Path(OUTPUT_DIR)
    wav_dir = out_dir / "wavs"
    wav_dir.mkdir(parents=True, exist_ok=True)

    failed_log_path = out_dir / "upload_failures.log"

    # Ensure the HF repo exists
    api.create_repo(repo_id=HF_UPLOAD_REPO, repo_type="dataset", token=HF_TOKEN, exist_ok=True)

    max_sec = max_hours * 3600 if max_hours else None
    manifest_paths, all_repo_files = get_manifest_paths(video_filter)

    # Resume: check HF remote, not local file
    done_ids = get_done_ids_from_hf()
    manifest_paths = [
        p for p in manifest_paths
        if os.path.basename(p).replace("_asr.json", "") not in done_ids
    ]
    print(f"📋 Processing {len(manifest_paths)} new videos. (Target: {max_hours or 'Unlimited'} hours)")

    total_sec = 0.0
    stats = {"videos": 0, "chunks": 0, "upload_ok": 0, "upload_fail": 0}

    for path in tqdm(manifest_paths, desc="Processing Videos"):
        if max_sec and total_sec >= max_sec:
            break

        video_id   = os.path.basename(path).replace("_asr.json", "")
        audio_path = find_audio_path(video_id, all_repo_files)
        if not audio_path:
            print(f"⚠️  No audio for {video_id}")
            continue

        try:
            manifest = download_json(path)
            audio    = load_audio(audio_path)
            meta_rows, added = chunk_and_save(
                audio, manifest["segments"], video_id, wav_dir, total_sec, max_sec
            )

            if not meta_rows:
                continue

            print(f"   ⬆️  Uploading {len(meta_rows)} chunks for {video_id}...")
            ok = upload_video_chunks(video_id, wav_dir, meta_rows, failed_log_path)

            if ok:
                stats["upload_ok"] += 1
                print(f"   ✅ {video_id} — {len(meta_rows)} chunks uploaded & verified")
            else:
                stats["upload_fail"] += 1
                print(f"   ⚠️  {video_id} — some chunks failed (logged)")

            append_metadata(meta_rows, out_dir)
            upload_master_metadata(out_dir)

            total_sec += added
            stats["videos"] += 1
            stats["chunks"] += len(meta_rows)

        except Exception as e:
            print(f"❌ Error on {video_id}: {e}")

    retry_failed_uploads(failed_log_path)

    print(f"""
╔══════════════════════════════════╗
║           Summary                ║
╠══════════════════════════════════╣
║  Videos processed : {stats['videos']:<13}║
║  Total chunks     : {stats['chunks']:<13}║
║  Duration         : {total_sec/3600:<10.2f}h   ║
║  Upload ✅        : {stats['upload_ok']:<13}║
║  Upload ❌        : {stats['upload_fail']:<13}║
╚══════════════════════════════════╝
""")

if __name__ == "__main__":
    import sys
    is_notebook = 'ipykernel' in sys.modules or 'google.colab' in sys.modules
    parser = argparse.ArgumentParser(description="Chunk ASR dataset and upload to HuggingFace on the fly")
    parser.add_argument("--videos",     nargs="+",  help="Specific Video IDs to process")
    parser.add_argument("--max_hours",  type=float, help="Stop after this many hours of audio")
    args, _ = parser.parse_known_args() if is_notebook else (parser.parse_args(), None)
    main(video_filter=args.videos, max_hours=args.max_hours)

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


🔍 Scanning frovolts/ssdata/asr_train_ready for manifests...
   ✅ Found 7 manifests.
   🆕 No remote metadata found, starting fresh.
📋 Processing 7 new videos. (Target: Unlimited hours)


Processing Videos:   0%|          | 0/7 [00:00<?, ?it/s]

7eEUCYhFxeo_asr.json: 0.00B [00:00, ?B/s]

data/audio/7eEUCYhFxeo.wav:   0%|          | 0.00/49.8M [00:00<?, ?B/s]

...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

   ✅ wLR0crYMXUk — 20 chunks uploaded & verified


Processing Videos: 100%|██████████| 7/7 [00:45<00:00,  6.51s/it]


╔══════════════════════════════════╗
║           Summary                ║
╠══════════════════════════════════╣
║  Videos processed : 7            ║
║  Total chunks     : 326          ║
║  Duration         : 2.07      h   ║
║  Upload ✅        : 7            ║
║  Upload ❌        : 0            ║
╚══════════════════════════════════╝



In [ ]:
#!/usr/bin/env python3
import requests
import json
import os
import time
from datetime import datetime
from pathlib import Path
from huggingface_hub import list_repo_files, HfApi

# --- CONFIGURATION ---
API_KEY = "```YOUR_PYANNOTE_API"
HF_TOKEN = secret_value_0 # Must be a 'READ' or 'WRITE' token
REPO_ID = "frovolts/ssdata"

# Paths
OUTPUT_DIR = '/kaggle/working/long_form_diarization_results'
REMOTE_FOLDER = "diarization_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# API and URL Constants
BASE_URL = "https://api.pyannote.ai/v1"
HF_RESOLVE_BASE = f"https://huggingface.co/datasets/{REPO_ID}/resolve/main"

def run_big_seven_pipeline():
    api = HfApi()
    print(f"🔍 Syncing with HF: {REPO_ID}...")
    
    try:
        all_files = list_repo_files(REPO_ID, repo_type="dataset", token=HF_TOKEN)
        
        # 1. Map Target IDs from ASR folder
        target_ids = {
            f.split('/')[-1].replace('_asr.json', '') 
            for f in all_files if 'asr_train_ready/' in f and f.endswith('.json')
        }
        
        # 2. Map Long-form Audio (ignoring chunks)
        audio_map = {}
        for f in all_files:
            v_id = f.split('/')[-1].rsplit('.', 1)[0]
            if v_id in target_ids and "chunk" not in f.lower():
                if any(f.lower().endswith(ext) for ext in ['.mp3', '.wav', '.m4a']):
                    audio_map[v_id] = f

        # 3. Filter Queue
        processed = {f.replace('_output.json', '') for f in os.listdir(OUTPUT_DIR)}
        queue = {v_id: path for v_id, path in audio_map.items() if v_id not in processed}

        print(f"✅ Found {len(audio_map)} Long-form files. Remaining to Task: {len(queue)}")
        if not queue:
            print("Nothing to do. Everything is already processed or uploaded.")
            return

        # --- PHASE 1: SUBMISSION ---
        active_jobs = []
        print("\n🚀 Submitting jobs with Authenticated URLs...")
        for i, (v_id, full_path) in enumerate(queue.items(), 1):
            # THE FIX: Append the token so Pyannote's server can bypass the private repo gate
            audio_url = f"{HF_RESOLVE_BASE}/{full_path}?token={HF_TOKEN}"
            
            headers = {"Authorization": f"Bearer {API_KEY}"}
            print(f"[{i}/{len(queue)}] Submitting {v_id}...", end=' ', flush=True)
            
            try:
                res = requests.post(f"{BASE_URL}/diarize", headers=headers, json={"url": audio_url})
                if res.status_code == 200:
                    active_jobs.append({"v_id": v_id, "job_id": res.json()["jobId"]})
                    print("✅")
                else:
                    print(f"❌ Error {res.status_code}: {res.text}")
            except Exception as e:
                print(f"❌ Connection error: {e}")
            
            time.sleep(3) # Politeness delay

        # --- PHASE 2: POLLING WITH HEARTBEAT ---
        print("\n⏳ Monitoring Pyannote processing (Heartbeat enabled)...")
        start_time = time.time()
        
        while active_jobs:
            elapsed = int(time.time() - start_time) // 60
            print(f"\n[Minute {elapsed}] Status Check:")
            
            for job in active_jobs[:]:
                try:
                    res = requests.get(f"{BASE_URL}/jobs/{job['job_id']}", 
                                       headers={"Authorization": f"Bearer {API_KEY}"}).json()
                    status = res.get('status')
                    
                    if status == 'succeeded':
                        out_path = os.path.join(OUTPUT_DIR, f"{job['v_id']}_output.json")
                        with open(out_path, 'w', encoding='utf-8') as f:
                            json.dump(res.get('output', {}), f, indent=2)
                        print(f"  ✨ {job['v_id']}: FINISHED")
                        active_jobs.remove(job)
                    elif status in ['processing', 'queued']:
                        print(f"  ⚙️ {job['v_id']}: {status.upper()}")
                    elif status in ['failed', 'canceled']:
                        # Capture why it failed (usually 'file too large' or 'download failed')
                        reason = res.get('error', 'Unknown Error')
                        print(f"  💀 {job['v_id']}: FAILED - Reason: {reason}")
                        active_jobs.remove(job)
                except Exception as e:
                    print(f"  ⚠️ {job['v_id']}: Polling error... {e}")
                    continue
            
            if active_jobs:
                time.sleep(30) # Poll every 30 seconds

        # --- PHASE 3: AUTOMATIC UPLOAD ---
        final_files = os.listdir(OUTPUT_DIR)
        if final_files:
            print(f"\n🚀 Uploading {len(final_files)} results to {REPO_ID}...")
            try:
                api.upload_folder(
                    folder_path=OUTPUT_DIR,
                    path_in_repo=REMOTE_FOLDER,
                    repo_id=REPO_ID,
                    repo_type="dataset",
                    token=HF_TOKEN,
                    commit_message=f"Reproducible Diarization Results: {len(final_files)} files"
                )
                print("✅ Upload successful. Reproducibility achieved.")
            except Exception as e:
                print(f"❌ Upload failed: {e}")
        else:
            print("\n❌ No files were successfully processed to upload.")

    except Exception as e:
        print(f"❌ Master Pipeline Failure: {e}")

if __name__ == "__main__":
    run_big_seven_pipeline()

🔍 Syncing with HF: frovolts/ssdata...
✅ Found 7 Long-form files. Remaining to Task: 7

🚀 Submitting jobs with Authenticated URLs...
[1/7] Submitting 7eEUCYhFxeo... ✅
[2/7] Submitting BRaTjnoRpdg... ✅
[3/7] Submitting PcQZTx-waB4... ✅
[4/7] Submitting X020uvCGuN8... ✅
[5/7] Submitting hY90r2kiAkE... ✅
[6/7] Submitting kezGvnt2tlk... ✅
[7/7] Submitting wLR0crYMXUk... ✅

⏳ Monitoring Pyannote processing (Heartbeat enabled)...

[Minute 0] Status Check:
  ✨ 7eEUCYhFxeo: FINISHED
  ✨ BRaTjnoRpdg: FINISHED

[Minute 0] Status Check:
  ✨ PcQZTx-waB4: FINISHED
  ✨ hY90r2kiAkE: FINISHED
  ✨ kezGvnt2tlk: FINISHED
  ✨ wLR0crYMXUk: FINISHED

[Minute 1] Status Check:
  ✨ X020uvCGuN8: FINISHED

🚀 Uploading 7 results to frovolts/ssdata...
✅ Upload successful. Reproducibility achieved.


In [9]:
import os

def print_tree(start_path, prefix=""):
    """Recursively prints directory structure in ASCII format."""
    try:
        items = sorted(os.listdir(start_path))
    except FileNotFoundError:
        print(f"Error: Path '{start_path}' not found.")
        return
    except PermissionError:
        print(f"Error: Permission denied for '{start_path}'.")
        return

    for index, item in enumerate(items):
        path = os.path.join(start_path, item)
        connector = "└── " if index == len(items) - 1 else "├── "
        print(prefix + connector + item)
        if os.path.isdir(path):
            extension = "    " if index == len(items) - 1 else "│   "
            print_tree(path, prefix + extension)

# Example usage:
if __name__ == "__main__":
    dataset_path = "/kaggle/working/"  # Change to your dataset folder
    print(f"Dataset structure for: {dataset_path}")
    print_tree(dataset_path)

Dataset structure for: /kaggle/working/
├── __notebook__.ipynb
├── asr_workspace
│   ├── 7eEUCYhFxeo_asr.json
│   ├── BRaTjnoRpdg_asr.json
│   ├── PcQZTx-waB4_asr.json
│   ├── X020uvCGuN8_asr.json
│   ├── hY90r2kiAkE_asr.json
│   ├── kezGvnt2tlk_asr.json
│   └── wLR0crYMXUk_asr.json
├── long_form_diarization_results
│   ├── 7eEUCYhFxeo_output.json
│   ├── BRaTjnoRpdg_output.json
│   ├── PcQZTx-waB4_output.json
│   ├── X020uvCGuN8_output.json
│   ├── hY90r2kiAkE_output.json
│   ├── kezGvnt2tlk_output.json
│   └── wLR0crYMXUk_output.json
├── videos_with_bangla.csv
└── wavs_asr_chunks
    ├── metadata.csv
    └── wavs
